# Almond Tree Segmentation Pipeline (Improved)

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Augmentation parameters
IMAGE_SIZE = 1536
BATCH_SIZE = 1

# --- Static augmentations (always applied) ---
STATIC_AUGS = [
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),
    A.OneOf([
        A.RandomShadow(p=0.3),
        A.RandomSunFlare(p=0.2),
        A.RandomFog(p=0.1),
    ], p=0.5),
    A.CLAHE(p=0.3),
    A.GridDistortion(p=0.2),
]

def get_dynamic_color_aug(image):
    """Dynamically set brightness/contrast limits based on image color stats."""
    means = image.mean(axis=(0, 1)) / 255.0
    stds = image.std(axis=(0, 1)) / 255.0
    brightness_limit = 0.1 if means.mean() > 0.6 else 0.4
    contrast_limit = 0.1 if stds.mean() > 0.2 else 0.3
    return [
        A.RandomBrightnessContrast(
            brightness_limit=brightness_limit,
            contrast_limit=contrast_limit,
            p=0.5
        ),
    ]

class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, img_info['file_name'])
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))

        # Combine static and dynamic augmentations
        dynamic_color_aug = get_dynamic_color_aug(image)
        full_transform = A.Compose(
            STATIC_AUGS + dynamic_color_aug + [
                A.Normalize(
                    mean=[0.348, 0.411, 0.324],
                    std=[0.145, 0.141, 0.135]
                ),
                ToTensorV2()
            ]
        )

        augmented = full_transform(image=image, mask=mask)
        image = augmented['image']
        mask = augmented['mask'].unsqueeze(0).float()

        return image, mask

# Validation transform (no augmentation, just resize and normalize)
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(
        mean=[0.348, 0.411, 0.324],
        std=[0.145, 0.141, 0.135]
    ),
    ToTensorV2()
])

### Transformations

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGE_SIZE = 1024  # or your preferred size

# Training augmentations
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7
    ),
    A.OneOf([
        A.RandomShadow(p=0.3),
        A.RandomSunFlare(p=0.2),
        A.RandomFog(p=0.1),
    ], p=0.5),
    A.CLAHE(p=0.3),
    A.GridDistortion(p=0.2),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225],   # ImageNet std
        max_pixel_value=255.0
    ),
    ToTensorV2()
])

# Validation/test augmentations (no randomness)
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225],   # ImageNet std
        max_pixel_value=255.0
    ),
    ToTensorV2()
])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Use the same IMAGE_SIZE and get_dynamic_color_aug as in your dataset code ---
IMAGE_SIZE = 1536

STATIC_AUGS = [
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.7),
    A.OneOf([
        A.RandomShadow(p=0.3),
        A.RandomSunFlare(p=0.2),
        A.RandomFog(p=0.1),
    ], p=0.5),
    A.CLAHE(p=0.3),
    A.GridDistortion(p=0.2),
]

def get_dynamic_color_aug(image):
    means = image.mean(axis=(0, 1)) / 255.0
    stds = image.std(axis=(0, 1)) / 255.0
    brightness_limit = 0.1 if means.mean() > 0.6 else 0.4
    contrast_limit = 0.1 if stds.mean() > 0.2 else 0.3
    return [
        A.RandomBrightnessContrast(
            brightness_limit=brightness_limit,
            contrast_limit=contrast_limit,
            p=0.5
        ),
    ]

# Load a few raw images from your training set folder (update the path)
image_paths = sorted(glob.glob("/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train/*.jpg"))[:5]  # Limit to first 5

# For visualization: no normalization or tensor conversion, just augmentation
def show_augmented_images(image_paths):
    for img_path in image_paths:
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Compose the full transform for this image
        dynamic_color_aug = get_dynamic_color_aug(image)
        full_transform = A.Compose(
            STATIC_AUGS + dynamic_color_aug
        )

        # Apply transforms
        orig_img = A.Resize(IMAGE_SIZE, IMAGE_SIZE)(image=image)["image"]
        aug_img = full_transform(image=image)["image"]

        # Plot original vs augmented
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(orig_img)
        plt.title("Original")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(np.clip(aug_img, 0, 255).astype(np.uint8))
        plt.title("Augmented (Static + Dynamic)")
        plt.axis("off")

        plt.show()

# Run it
show_augmented_images(image_paths)


### Transformations

### Paths Datasets and Dataloaders

In [ ]:
# Paths
train_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# For training: do NOT pass a transform, so dynamic transform is built per image
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path)

# For validation: pass the static transform
val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, val_transform)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


### Model

In [ ]:
# 1. Enhanced Model Architecture with Spatial Attention
model = smp.Unet(
    encoder_name="efficientnet-b7",       # Deeper encoder for aerial details
    encoder_weights="imagenet",
    decoder_attention_type="scse",        # Spatial-channel attention
    in_channels=3,
    classes=1,
    activation=None
).to(device)

In [ ]:
import torch.nn.functional as F

# 3. Advanced Loss Function
class TreeSegmentationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary')
        self.focal = smp.losses.FocalLoss(mode='binary')
        self.lovasz = smp.losses.LovaszLoss(mode='binary')
        
    def forward(self, inputs, targets):
        return (
            0.5 * self.dice(inputs, targets) +
            0.3 * self.focal(inputs, targets) +
            0.2 * self.lovasz(inputs, targets)
        )

loss_fn = TreeSegmentationLoss()

# 4. Optimizer Configuration
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,                   # Lower initial learning rate
    weight_decay=1e-5          # Regularization
)

# Scheduler with fixed closing parenthesis
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True)

In [ ]:
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

### Training and Evaluation

In [ ]:
# Initialize history tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': []
}

# Best score tracking
best_val_dice = 0
patience = 10
epochs_no_improve = 0

# Scheduler for dynamic learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

# Main training loop
for epoch in range(1, 51):
    model.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_dice += dice_coef(outputs, masks).item()
        train_iou += iou_score(outputs, masks).item()

    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            processed = postprocess_mask(outputs)
            val_loss += loss_fn(outputs, masks).item()
            val_dice += dice_coef(outputs, masks).item()
            val_iou += iou_score(outputs, masks).item()

    # Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_train_iou = train_iou / len(train_loader)
    avg_val_iou = val_iou / len(val_loader)

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Logging
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | "
          f"Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f}")

    # Check for improvement
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        epochs_no_improve = 0

        # ✅ Full checkpoint saving
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': avg_val_dice,
            'val_loss': avg_val_loss
        }, "best_model.pth")

        print("✅ Saved new best model with optimizer and metrics")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve} epochs")

    # ⛔ Early stopping
    if epochs_no_improve >= patience:
        print("⛔ Early stopping triggered")
        break

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)


In [ ]:
plot_learning_curves(history)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Define test transform
test_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE), 
    A.Normalize(),  # Uses ImageNet stats
    ToTensorV2(),
])

# 2. Load test dataset
test_dataset = COCOSegmentationDataset(
    img_dir="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/",  
    ann_path="/Users/snirtahasa/Desktop/Almonds - Project/Almons-Trees-3/test/_annotations.coco.json", 
    transform=test_transform
)

# 3. Recreate the model with the same architecture used during training
model = smp.Unet(
    encoder_name="efficientnet-b3",     # Same as training
    encoder_weights=None,               # No pretrained weights during loading
    in_channels=3,
    classes=1,
    activation=None                     # Must match training config
).to(device)

# 4. Load the full checkpoint and restore the model weights
checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 5. Denormalize function for visualization
def denormalize_image(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    if isinstance(tensor_img, torch.Tensor):
        tensor_img = tensor_img.detach().cpu()
    np_img = tensor_img.permute(1, 2, 0).numpy()
    np_img = np_img * np.array(std) + np.array(mean)
    np_img = np.clip(np_img, 0, 1)
    np_img = (np_img * 255).astype(np.uint8)
    return np_img

# 6. Prediction + Visualization
def visualize_predictions(model, dataset, device, max_samples=None):
    n_samples = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in tqdm(range(n_samples), desc="Predicting"):
        image, true_mask = dataset[i]
        image_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            pred_mask = model(image_tensor)

        pred_mask = (pred_mask.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
        image_np = denormalize_image(image)
        true_mask_np = true_mask.squeeze().cpu().numpy()

        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        axs[0].imshow(image_np)
        axs[0].set_title("Original Image")
        axs[1].imshow(true_mask_np, cmap='gray')
        axs[1].set_title("Ground Truth")
        axs[2].imshow(pred_mask, cmap='gray')
        axs[2].set_title("Predicted Mask")
        for ax in axs:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

# 7. Run predictions
visualize_predictions(model, test_dataset, device, max_samples=10)
